# PixArt-Sigma 512: Image Archive → Clean Latents → Reconstruction Validation

This Colab notebook prepares image features from the local ink-painting dataset for `jackyckp/efficient-pixart-sigma-lora`. It will:

1. Mount Google Drive, clone the GitHub repository, and automatically locate a `.zip` archive;
2. Extract the dataset and recursively pair images with same-named `.txt` caption files;
3. Create a deterministically sorted `manifest.jsonl` with the `sample_id` values needed to align the image cache with the T5 embedding cache;
4. Use the SDXL VAE bundled with PixArt-Sigma to encode each preprocessed 512×512 image as a clean latent with shape `[4, 64, 64]`;
5. Save a single `.pt` cache and decode selected latents for reconstruction comparison.

This notebook **does not add diffusion noise**. Random timesteps and Gaussian noise must be generated dynamically inside the LoRA training loop.

> In Colab, select `Runtime → Change runtime type → T4 GPU`, then run every cell from top to bottom. Outputs are saved to Google Drive by default so they survive runtime disconnections.


In [ ]:
#@title 1. Install dependencies (about 1–2 minutes on the first run)
%pip install -q "diffusers==0.39.0" "transformers>=4.46,<6" "accelerate>=1.2" "safetensors>=0.5" "Pillow>=10" "tqdm>=4.66" "matplotlib>=3.8"


In [ ]:
#@title 2. Configure paths, mount Drive, and check the GPU
from google.colab import drive
drive.mount("/content/drive")

import gc
import hashlib
import json
import os
import random
import shutil
import subprocess
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from diffusers import AutoencoderKL
from PIL import Image, ImageOps
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "No CUDA GPU was detected. Select a T4 GPU from the Runtime menu, then run this cell again."

# Keep these settings consistent with the repository's pixart_sigma_lora_smoke_test_official.py.
REPO_URL = "https://github.com/jackyckp/efficient-pixart-sigma-lora.git"
REPO_DIR = Path("/content/efficient-pixart-sigma-lora")
COMPONENT_MODEL = "PixArt-alpha/pixart_sigma_sdxlvae_T5_diffusers"
TRANSFORMER_MODEL = "PixArt-alpha/PixArt-Sigma-XL-2-512-MS"  # Recorded in metadata only; this notebook does not load the DiT
RESOLUTION = 512
SEED = 42
BATCH_SIZE = 4  # Usually works on a 16 GB T4; reduce to 2 or 1 if an OOM error occurs
EXPECTED_NUM_IMAGES = 260  # Change to 300 when the dataset is expanded; set to None to disable the count check
FORCE_REBUILD = False

# If the repository has no ZIP archive, place it in Drive and set DATA_ZIP_OVERRIDE to its path.
# Example: Path("/content/drive/MyDrive/pixart_sigma/ink.zip")
DATA_ZIP_OVERRIDE = None

OUTPUT_DIR = Path("/content/drive/MyDrive/pixart_sigma/clean_latents_512")
EXTRACT_DIR = Path("/content/pixart_ink_extracted")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name} ({vram_gb:.1f} GiB)")
print(f"Output: {OUTPUT_DIR}")


## Dataset Input

By default, the notebook clones the repository and then:

- Uses the archive automatically if the repository contains exactly one `.zip` file;
- If multiple `.zip` files exist, prefers the single archive whose filename contains `ink`; otherwise, it asks you to specify one explicitly in the configuration;
- Reads `data/ink` directly if the repository contains no `.zip` file but already has that directory;
- Allows you to specify an archive in Google Drive through `DATA_ZIP_OVERRIDE`.

Extraction takes place only in Colab's temporary `/content` storage. The final cache, manifest, and reconstruction images are written to Google Drive.


In [ ]:
#@title 3. Clone the repository, locate the archive, and extract the dataset
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Reuse existing repository: {REPO_DIR}")

def choose_zip(repo_dir, override=None):
    if override is not None:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f"DATA_ZIP_OVERRIDE does not exist: {path}")
        return path
    candidates = sorted(p for p in repo_dir.rglob("*.zip") if p.is_file())
    if len(candidates) == 1:
        return candidates[0]
    ink_candidates = [p for p in candidates if "ink" in p.name.lower()]
    if len(ink_candidates) == 1:
        return ink_candidates[0]
    if len(candidates) > 1:
        raise RuntimeError(
            "Multiple ZIP archives were found, so the dataset cannot be selected safely. Set DATA_ZIP_OVERRIDE to one of these files:\n"
            + "\n".join(str(p) for p in candidates)
        )
    return None

data_zip = choose_zip(REPO_DIR, DATA_ZIP_OVERRIDE)
if data_zip is not None:
    if EXTRACT_DIR.exists() and FORCE_REBUILD:
        shutil.rmtree(EXTRACT_DIR)
    if not EXTRACT_DIR.exists():
        EXTRACT_DIR.mkdir(parents=True)
        with zipfile.ZipFile(data_zip) as archive:
            bad_member = archive.testzip()
            if bad_member is not None:
                raise zipfile.BadZipFile(f"The ZIP archive is corrupted. First invalid member: {bad_member}")
            archive.extractall(EXTRACT_DIR)
    DATA_SEARCH_ROOT = EXTRACT_DIR
    print(f"Zip: {data_zip}")
    print(f"Extracted to: {EXTRACT_DIR}")
elif (REPO_DIR / "data" / "ink").is_dir():
    DATA_SEARCH_ROOT = REPO_DIR / "data" / "ink"
    print("No zip found; using repository data/ink directly.")
else:
    raise FileNotFoundError(
        "No ZIP archive or data/ink directory was found. Put the archive in Drive and set DATA_ZIP_OVERRIDE."
    )


In [ ]:
#@title 4. Recursively pair images and captions, then build a deterministic manifest
SUPPORTED_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

def natural_key(path):
    import re
    parts = re.split(r"(\d+)", str(path).replace("\\", "/").lower())
    return tuple(int(part) if part.isdigit() else part for part in parts)

all_images = sorted(
    (p for p in DATA_SEARCH_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in SUPPORTED_EXTS),
    key=natural_key,
)
paired_images = [p for p in all_images if p.with_suffix(".txt").is_file()]
missing_caption = [p for p in all_images if not p.with_suffix(".txt").is_file()]
orphan_captions = [
    p for p in DATA_SEARCH_ROOT.rglob("*.txt")
    if not any(p.with_suffix(ext).is_file() for ext in SUPPORTED_EXTS)
]

if not paired_images:
    raise RuntimeError(f"No same-named image and .txt caption pairs were found under {DATA_SEARCH_ROOT}.")

# Find the common parent of all paired images. For paths such as ink/animal and ink/plant, this resolves to ink.
dataset_root = Path(os.path.commonpath([str(p.parent) for p in paired_images]))
records = []
for image_path in paired_images:
    caption_path = image_path.with_suffix(".txt")
    caption = caption_path.read_text(encoding="utf-8-sig").strip()
    if not caption:
        raise ValueError(f"Caption is empty: {caption_path}")
    relative_image = image_path.relative_to(dataset_root)
    sample_id = relative_image.with_suffix("").as_posix()
    with Image.open(image_path) as image:
        width, height = image.size
    records.append({
        "sample_id": sample_id,
        "image_path": image_path,
        "caption_path": caption_path,
        "relative_image_path": relative_image.as_posix(),
        "relative_caption_path": caption_path.relative_to(dataset_root).as_posix(),
        "caption": caption,
        "original_width": width,
        "original_height": height,
    })

sample_ids = [row["sample_id"] for row in records]
if len(sample_ids) != len(set(sample_ids)):
    duplicates = sorted({x for x in sample_ids if sample_ids.count(x) > 1})
    raise ValueError(f"Duplicate sample_id values: {duplicates[:10]}")
if missing_caption or orphan_captions:
    raise ValueError(
        f"Incomplete pairs: {len(missing_caption)} images are missing captions, and "
        f"{len(orphan_captions)} captions are missing images. Fix the dataset before continuing."
    )
if EXPECTED_NUM_IMAGES is not None and len(records) != EXPECTED_NUM_IMAGES:
    raise ValueError(
        f"Expected {EXPECTED_NUM_IMAGES} pairs, but found {len(records)}. "
        "If the dataset was intentionally updated, change EXPECTED_NUM_IMAGES and run the notebook again."
    )

manifest_path = OUTPUT_DIR / "manifest.jsonl"
with manifest_path.open("w", encoding="utf-8") as handle:
    for row in records:
        public_row = {key: value for key, value in row.items() if key not in {"image_path", "caption_path"}}
        handle.write(json.dumps(public_row, ensure_ascii=False) + "\n")

category_counts = {}
for sample_id in sample_ids:
    category = sample_id.split("/", 1)[0] if "/" in sample_id else "root"
    category_counts[category] = category_counts.get(category, 0) + 1

print(f"Dataset root : {dataset_root}")
print(f"Valid pairs  : {len(records)}")
print(f"Categories   : {category_counts}")
print(f"Manifest     : {manifest_path}")
print("First IDs   :", sample_ids[:5])


## Fixed Image Preprocessing

To stay consistent with the repository's existing smoke test, this notebook uses:

```text
RGB → aspect-ratio-preserving resize → 512×512 center crop → map to [-1, 1]
```

This process crops the edges of non-square images. The notebook displays several crop previews first; **confirm that the main subjects are not severely cropped before encoding the full dataset**. If the team later switches to padding, random cropping, or aspect-ratio bucketing, both the image latents and the training data loader must be regenerated together. Do not mix caches created with different preprocessing methods.


In [ ]:
#@title 5. Preview the center crops
def preprocess_pil(path):
    with Image.open(path) as image:
        return ImageOps.fit(
            image.convert("RGB"),
            (RESOLUTION, RESOLUTION),
            method=Image.Resampling.LANCZOS,
            centering=(0.5, 0.5),
        )

def pil_to_tensor(image):
    array = np.asarray(image, dtype=np.float32).copy() / 127.5 - 1.0
    return torch.from_numpy(array).permute(2, 0, 1).contiguous()

preview_indices = np.linspace(0, len(records) - 1, min(6, len(records)), dtype=int).tolist()
fig, axes = plt.subplots(2, len(preview_indices), figsize=(3 * len(preview_indices), 6))
for column, index in enumerate(preview_indices):
    with Image.open(records[index]["image_path"]) as original:
        axes[0, column].imshow(original.convert("RGB"))
    axes[0, column].set_title(f"Original\n{records[index]['sample_id']}")
    axes[1, column].imshow(preprocess_pil(records[index]["image_path"]))
    axes[1, column].set_title("Center crop 512")
    axes[0, column].axis("off")
    axes[1, column].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
#@title 6. Load the SDXL VAE bundled with PixArt-Sigma
vae = AutoencoderKL.from_pretrained(
    COMPONENT_MODEL,
    subfolder="vae",
    torch_dtype=torch.float16,
    use_safetensors=True,
).eval().to("cuda")
vae.requires_grad_(False)

scaling_factor = float(vae.config.scaling_factor)
vae_scale_factor = 2 ** (len(vae.config.block_out_channels) - 1)
latent_size = RESOLUTION // vae_scale_factor

assert vae.config.latent_channels == 4
assert vae_scale_factor == 8
assert latent_size == 64
print(f"VAE model       : {COMPONENT_MODEL}/vae")
print(f"Scaling factor  : {scaling_factor}")
print(f"Expected latent : [N, 4, {latent_size}, {latent_size}]")


In [ ]:
#@title 7. Run a smoke test on five images
smoke_records = records[:5]
smoke_pixels = torch.stack([
    pil_to_tensor(preprocess_pil(row["image_path"])) for row in smoke_records
]).to("cuda", dtype=torch.float16)

smoke_generator = torch.Generator(device="cuda").manual_seed(SEED)
with torch.inference_mode():
    smoke_latents = vae.encode(smoke_pixels).latent_dist.sample(generator=smoke_generator)
    smoke_latents = smoke_latents * scaling_factor
    smoke_reconstruction = vae.decode(smoke_latents / scaling_factor).sample

assert smoke_latents.shape == (len(smoke_records), 4, latent_size, latent_size)
assert torch.isfinite(smoke_latents).all()
print("PASS — smoke latent shape:", tuple(smoke_latents.shape))
print("Latent dtype/min/max/mean/std:", smoke_latents.dtype, smoke_latents.min().item(), smoke_latents.max().item(), smoke_latents.mean().item(), smoke_latents.std().item())

def tensor_to_pil(tensor):
    array = ((tensor.detach().float().cpu().clamp(-1, 1) + 1) * 127.5).round().byte()
    return Image.fromarray(array.permute(1, 2, 0).numpy())

fig, axes = plt.subplots(2, len(smoke_records), figsize=(3 * len(smoke_records), 6))
for column, row in enumerate(smoke_records):
    axes[0, column].imshow(preprocess_pil(row["image_path"]))
    axes[0, column].set_title(f"Input\n{row['sample_id']}")
    axes[1, column].imshow(tensor_to_pil(smoke_reconstruction[column]))
    axes[1, column].set_title("VAE reconstruction")
    axes[0, column].axis("off")
    axes[1, column].axis("off")
plt.tight_layout()
plt.show()

del smoke_pixels, smoke_latents, smoke_reconstruction
torch.cuda.empty_cache()


## Encode and Save All Clean Latents

The following cell encodes images in the fixed manifest order. `posterior.sample()` uses a seeded CUDA generator, making results reproducible under the same software and hardware execution path. The sampled output is still a clean latent \(x_0\), because no diffusion scheduler or additional Gaussian noise is used here.

Core fields in the output `.pt` file:

- `latents`: `[N, 4, 64, 64]` CPU `float16` tensor;
- `sample_ids`: in exactly the same order as the manifest;
- `relative_image_paths`: excludes temporary absolute Colab paths;
- `vae_model`, `scaling_factor`, `preprocessing`, and `seed`: metadata required for reproducibility.

The cache is first written to a temporary file and then atomically replaced to reduce the risk of leaving a partial cache if the runtime is interrupted.


In [ ]:
#@title 8. Generate and save all clean latents
manifest_fingerprint = hashlib.sha256(
    json.dumps(
        [(row["sample_id"], row["caption"], row["original_width"], row["original_height"]) for row in records],
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()[:12]

cache_path = OUTPUT_DIR / f"image_latents_n{len(records)}_res{RESOLUTION}_{manifest_fingerprint}.pt"
temp_cache_path = cache_path.with_suffix(".tmp")

if cache_path.exists() and not FORCE_REBUILD:
    print(f"Cache already exists; skip encoding: {cache_path}")
    latent_cache = torch.load(cache_path, map_location="cpu", weights_only=True)
else:
    all_latents = []
    generator = torch.Generator(device="cuda").manual_seed(SEED)
    for start in tqdm(range(0, len(records), BATCH_SIZE), desc="Encoding clean latents"):
        batch_records = records[start:start + BATCH_SIZE]
        pixels = torch.stack([
            pil_to_tensor(preprocess_pil(row["image_path"])) for row in batch_records
        ]).to("cuda", dtype=torch.float16)
        with torch.inference_mode():
            posterior = vae.encode(pixels).latent_dist
            batch_latents = posterior.sample(generator=generator) * scaling_factor
        all_latents.append(batch_latents.cpu().to(torch.float16).contiguous())
        del pixels, posterior, batch_latents

    clean_latents = torch.cat(all_latents, dim=0).contiguous()
    latent_cache = {
        "format_version": 1,
        "latents": clean_latents,
        "sample_ids": sample_ids,
        "relative_image_paths": [row["relative_image_path"] for row in records],
        # Compatibility aliases used by the repository's existing smoke-test cache schema.
        "image_paths": [row["relative_image_path"] for row in records],
        "num_images": len(records),
        "resolution": RESOLUTION,
        "latent_shape_per_image": [4, latent_size, latent_size],
        "latent_dtype": "float16",
        "latent_kind": "clean_x0_scaled",
        "vae_model": COMPONENT_MODEL,
        "vae_subfolder": "vae",
        "transformer_model": TRANSFORMER_MODEL,
        "scaling_factor": scaling_factor,
        "vae_scaling_factor": scaling_factor,
        "vae_scale_factor": vae_scale_factor,
        "latent_sampling": "posterior.sample",
        "seed": SEED,
        "preprocessing": {
            "convert": "RGB",
            "resize_crop": "PIL.ImageOps.fit",
            "size": [RESOLUTION, RESOLUTION],
            "resampling": "LANCZOS",
            "centering": [0.5, 0.5],
            "normalization": "pixel / 127.5 - 1.0",
        },
        "manifest_filename": manifest_path.name,
        "manifest_fingerprint": manifest_fingerprint,
    }
    torch.save(latent_cache, temp_cache_path)
    os.replace(temp_cache_path, cache_path)
    del all_latents, clean_latents

print(f"Saved/loaded: {cache_path}")
print(f"Size: {cache_path.stat().st_size / 1024**2:.1f} MiB")


In [ ]:
#@title 9. Validate the cache and save reconstruction comparisons
latents = latent_cache["latents"]
assert isinstance(latents, torch.Tensor)
assert latents.shape == (len(records), 4, latent_size, latent_size), latents.shape
assert latents.dtype == torch.float16
assert torch.isfinite(latents).all()
assert latent_cache["sample_ids"] == sample_ids
assert latent_cache["latent_kind"] == "clean_x0_scaled"
assert abs(float(latent_cache["scaling_factor"]) - scaling_factor) < 1e-12

# Sample uniformly across the sorted manifest instead of checking only one category.
recon_indices = np.linspace(0, len(records) - 1, min(8, len(records)), dtype=int).tolist()
recon_dir = OUTPUT_DIR / "reconstruction_examples"
recon_dir.mkdir(exist_ok=True)

decoded_batches = []
with torch.inference_mode():
    for start in range(0, len(recon_indices), BATCH_SIZE):
        selected_latents = latents[recon_indices[start:start + BATCH_SIZE]].to("cuda", dtype=torch.float16)
        decoded_batches.append(vae.decode(selected_latents / scaling_factor).sample.cpu())
decoded = torch.cat(decoded_batches, dim=0)

fig, axes = plt.subplots(2, len(recon_indices), figsize=(3 * len(recon_indices), 6))
for column, (index, decoded_tensor) in enumerate(zip(recon_indices, decoded)):
    input_image = preprocess_pil(records[index]["image_path"])
    reconstructed_image = tensor_to_pil(decoded_tensor)
    safe_id = records[index]["sample_id"].replace("/", "__")
    input_image.save(recon_dir / f"{safe_id}__input.png")
    reconstructed_image.save(recon_dir / f"{safe_id}__reconstruction.png")
    axes[0, column].imshow(input_image)
    axes[0, column].set_title(f"Input\n{records[index]['sample_id']}")
    axes[1, column].imshow(reconstructed_image)
    axes[1, column].set_title("Reconstruction")
    axes[0, column].axis("off")
    axes[1, column].axis("off")
plt.tight_layout()
grid_path = recon_dir / "reconstruction_grid.png"
plt.savefig(grid_path, dpi=160, bbox_inches="tight")
plt.show()

summary = {
    "status": "PASS",
    "cache_file": cache_path.name,
    "manifest_file": manifest_path.name,
    "num_images": len(records),
    "latents_shape": list(latents.shape),
    "latents_dtype": str(latents.dtype),
    "all_finite": bool(torch.isfinite(latents).all()),
    "latent_min": float(latents.min()),
    "latent_max": float(latents.max()),
    "latent_mean": float(latents.float().mean()),
    "latent_std": float(latents.float().std()),
    "scaling_factor": scaling_factor,
    "manifest_fingerprint": manifest_fingerprint,
    "reconstruction_indices": recon_indices,
}
summary_path = OUTPUT_DIR / "validation_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps(summary, indent=2))
print("\nPASS — clean latent cache is ready for PixArt-Sigma LoRA training.")
print(f"Cache          : {cache_path}")
print(f"Manifest       : {manifest_path}")
print(f"Recon grid     : {grid_path}")
print(f"Validation JSON: {summary_path}")


## Loading the Cache During Training

```python
cache = torch.load(CACHE_PATH, map_location="cpu", weights_only=True)
clean_latents = cache["latents"]       # [N, 4, 64, 64]
sample_ids = cache["sample_ids"]       # Align with the text cache

assert sample_ids == text_cache["sample_ids"]

# Add noise dynamically at every training step; never save noisy_latents back to the cache.
noise = torch.randn_like(clean_latents_batch)
timesteps = torch.randint(
    0,
    noise_scheduler.config.num_train_timesteps,
    (clean_latents_batch.shape[0],),
    device=clean_latents_batch.device,
)
noisy_latents = noise_scheduler.add_noise(clean_latents_batch, noise, timesteps)
```

Give the same `manifest.jsonl` file to the teammate generating the T5 embeddings. Do not let the image and text pipelines scan and sort the dataset independently.
